## 1. Setup do ambiente

### 1.1 Importação de Bibliotecas
Importação de funções essenciais do **PySpark**, 
Selecionamos especificamente a função (`current_timestamp`) para a criação da coluna `data_criacao_silver` na tabela criada `chamados_geral`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, col, trim, regexp_replace, initcap, when, lower, lit, udf
from pyspark.sql.types import StringType, IntegerType, LongType, TimestampType, DecimalType

catalogo = 'medalhao_credit'
bronze_db_name = 'bronze_credit'
silver_db_name = 'silver_credit' 

### 1.2 Configuração de Ambiente

Definição do catálogo `catalogo` e o schema `silver_db_name` que serão utilizados. 

In [0]:
spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {silver_db_name}")

## 2. Tratamento da tabela `chamados_hora` 

### 2.1 Tabela `chamados_hora` na camada bronze

A tabela armazena informações sobre chamados de atendimento na camada bronze do Data Lake. A tabela contém os seguintes campos:

- **ID_Chamado**: Identificador único do chamado.
- **ID_Cliente**: Identificador do cliente relacionado ao chamado.
- **Hora_Abertura_Chamado**: Data e hora em que o chamado foi aberto.
- **Hora_Inicio_Atendimento**: Data e hora de início do atendimento do chamado.
- **Hora_Finalizacao_Atendimento**: Data e hora de finalização do atendimento.
- **data_ingestao**: Data de ingestão do registro na camada bronze.

In [0]:
df_ft_chamados_hora = spark.table(f'{catalogo}.{bronze_db_name}.ft_chamados_hora')
df_ft_chamados_hora.limit(5).display()

### 2.2 Tratamento de nomes das colunas na tabela `chamados_hora`

- Tratamento do header
- Os nomes das colunas do DataFrame foram convertidos para letras minúsculas, garantindo padronização.

In [0]:
nomes_colunas = [
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "data_ingestao"
                ]

df_ft_chamados_hora = df_ft_chamados_hora.toDF(*nomes_colunas)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora.limit(1))

df_ft_chamados_hora.limit(5).display()

### 2.3 Verificação / limpeza de dados da tabela `chamados_hora` 

- **Contagem inicial de linhas**: Verifica o número total de registros presentes no DataFrame `chamados_hora`.
- **Verificação de valores nulos**: Verifica linhas onde qualquer uma das colunas essenciais (`hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`, `data_ingestao`) possui valor nulo.
- **Verificação de linhas com tempos inconsistentes**: Verifica registros onde os cálculos de tempo (`tempo_espera_seg`, `tempo_atendimento_seg`, `diff_abertura_ingestao_seg`) resultam em valores negativos, indicando inconsistência temporal.
- **Verificação de valores irregulares em identificadores**: Verifica linhas onde os campos `id_cliente` e `id_chamado` são nulos ou não seguem o padrão numérico esperado.
- **Contagem final de linhas**: Exibe o total de registros restantes após todas as etapas de filtragem.

In [0]:
print(f'linhas em chamados_hora: {df_ft_chamados_hora.count()}')

In [0]:
df_ft_chamados_hora_null = df_ft_chamados_hora.filter(
    F.col('hora_abertura_chamado').isNull() |
    F.col('hora_inicio_atendimento').isNull() |
    F.col('hora_finalizacao_atendimento').isNull() |
    F.col('data_ingestao').isNull()
)

if df_ft_chamados_hora_null.count() == 0:
    print('valores nulos: 0')
else:
    print(f'linhas com val nulos: {df_ft_chamados_hora_null.count()}') 
    df_ft_chamados_hora_null.limit(5).display()

#### 2.3.1 Tratamento de valores na tabela `chamados_hora`

- As colunas `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento` passaram por duas etapas:
  1. Remoção de caracteres indesejados (" às ") usando `regexp_replace` para limpar os valores.
  2. Conversão dos valores dessas colunas para o tipo `timestamp`, utilizando o formato `'dd/MM/yyyy HH:mm:ss'`.

In [0]:
hora_cols = ['hora_abertura_chamado', 'hora_inicio_atendimento', 'hora_finalizacao_atendimento']

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.regexp_replace(F.col(col), r' �s ', ' ')
    )

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.to_timestamp(col, 'dd/MM/yyyy HH:mm:ss')
    )

#### 2.3.2 Criação de colunas na tabela `chamados_hora`

- **tempo_espera_seg**: tempo entre a abertura do chamado e o início do atendimento.
- **tempo_atendimento_seg**: tempo entre o início e a finalização do atendimento.
- **diff_abertura_ingestao_seg**: tempo entre a abertura do chamado e o momento de ingestão do registro na base.

Os cálculos são feitos convertendo os timestamps para o tipo `long` (segundos desde a época Unix) e subtraindo os valores correspondentes.

In [0]:
df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
    'tempo_espera_seg',
    (F.col('hora_inicio_atendimento').cast('long') - F.col('hora_abertura_chamado').cast('long'))
).withColumn(
    'tempo_atendimento_seg',
    (F.col('hora_finalizacao_atendimento').cast('long') - F.col('hora_inicio_atendimento').cast('long'))
).withColumn(
    'diff_abertura_ingestao_seg',
    (F.col('data_ingestao').cast('long') - F.col('hora_abertura_chamado').cast('long'))
)

In [0]:
df_ft_chamados_hora.limit(5).display()

In [0]:
# Verificação de linhas onde o tempo é inconsistente

df_ft_chamados_hora_tempo_dif = df_ft_chamados_hora.filter(
    (F.col('tempo_espera_seg') < 0) |
    (F.col('tempo_atendimento_seg') < 0) |
    (F.col('diff_abertura_ingestao_seg') < 0)
)

if df_ft_chamados_hora_tempo_dif.count() == 0:
    print('valores com tempo inconsistente: 0')
else:
    print(f'linhas com tempo inconsistente:')
    df_ft_chamados_hora_tempo_dif.limit(5).display()

In [0]:
# Verificação de valores irregulares em id_cliente e id_chamado

df_ft_chamados_hora_irreg = df_ft_chamados_hora.filter(
    F.col('id_cliente').isNull() &
    F.col('id_chamado').isNull() &
    ~F.col('id_cliente').rlike(r'^[0-9]+$') &
    ~F.col('id_chamado').rlike(r'^[0-9]+$')
)

if df_ft_chamados_hora_irreg.count() == 0:
    print('linhas com valores irregulares: 0')
else:
    print(f'linhas com id_cliente, id_chamado irregulares:')
    df_ft_chamados_hora_irreg.limit(5).display()

### 2.4 Remoção de registros inconsistentes

- Removemos da tabela (`df_ft_chamados_hora`) todos os registros com valores temporais inconsistentes.
- Essa etapa garante que apenas os chamados com dados temporais consistentes permaneçam para análise, eliminando registros com diferenças de tempo consideradas inválidas.

In [0]:
df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_tempo_dif)

print(f'linhas em chamados_hora apos remocao: {df_ft_chamados_hora.count()}')

### 2.5 Salvando dados tratados da tabela `chamados_hora` na camada silver

- O DataFrame `df_ft_chamados_hora`, após todas as etapas de limpeza e transformação, é salvo na tabela `chamados_hora` na camada silver utilizando o método `saveAsTable` com o modo `overwrite`, garantindo que os dados estejam atualizados.

In [0]:
df_ft_chamados_hora.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_hora')

df = spark.table(f'{catalogo}.{silver_db_name}.chamados_hora')
df.limit(5).display()

## 3. Tratamento da tabela `chamados`

### 3.1. Configuração e Leitura da Bronze
Defini as variáveis de ambiente `catalogo`, `bronze_db_name` e `silver_db_name` para organizar os caminhos do *Data Lake*.
Em seguida, realizei a leitura da tabela bruta (`df_bronze`). Como o arquivo original não possuía cabeçalho (gerando colunas genéricas como `_c0`), preparei o DataFrame para as transformações seguintes.

In [0]:
# Leitura da tabela Bronze de Chamados
df_bronze = spark.table(f"{catalogo}.{bronze_db_name}.ft_chamados")

display(df_bronze.limit(5))

In [0]:
# Célula de Diagnóstico
print("Lista exata de colunas:")
print(df_bronze.columns)

### 3.2. Tratamento de Identificadores (IDs)
Nesta etapa, foquei na chave primária da tabela. Renomeei a coluna genérica `_c0` para `id_chamado`, seguindo as boas práticas de *snake_case*.
Para garantir a integridade dos dados, converti o campo para o tipo Inteiro (`int`) e apliquei a remoção de duplicatas (`dropDuplicates`), assegurando que cada chamado seja único na camada Silver.

In [0]:
df_ordenado = (
    df_bronze
    .withColumnRenamed("_c0", "id_chamado") #Renomeia a coluna
    .withColumn("id_chamado", col("id_chamado").cast("int")) #Transforma tudo em int
    .filter(col("id_chamado").isNotNull())  #Remove linhas se o ID estiver vazio (lixo)
    .dropDuplicates(["id_chamado"])         #Se tiver dois IDs iguais, mantém apenas um
    .orderBy("id_chamado")
)

display(df_ordenado)

### 3.3. Normalização e Tipagem do ID Cliente
No dataframe `df_cliente_tratado`, renomeei a coluna para `id_cliente` (padrão *snake_case*).
Optei pela tipagem `long` para preservar a integridade de números grandes (como CPFs) e tratei os valores nulos preenchendo com `-1` (Cliente Desconhecido), garantindo que nenhum chamado fosse descartado por falta de identificação do cliente.

In [0]:
df_cliente_tratado = (
    df_ordenado # Continuando do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c1", "id_cliente")
    
    # 2. Tipagem SEGURA (Long em vez de Int para não quebrar CPFs)
    .withColumn("id_cliente", col("id_cliente").cast("long"))
    
    # 3. Tratamento de Nulos (Regra de Ouro)
    # Não apaga a linha (o chamado existiu), mas marca o cliente como -1 (Desconhecido)
    .fillna(-1, subset=["id_cliente"])
)

display(df_cliente_tratado)

### 3.4. Reconstrução da Coluna Motivo
No dataframe `df_motivo_tratado`, corrigi os erros de *encoding* (ex: "Contrata..o") utilizando Expressões Regulares (*Regex*).
Substituí os padrões corrompidos pelas palavras corretas e refinei a regra da palavra "Não" (usando `\b` para limites de palavra), evitando alterações indevidas em palavras como "Pontos". Finalizei padronizando o texto com a primeira letra maiúscula (*Initcap*).

In [0]:
df_motivo_tratado = (
    df_cliente_tratado # Continua do passo de ID_Cliente
    
    # 1. Renomear
    .withColumnRenamed("_c2", "motivo")
    
    # 2. Tipagem e Trim (Limpeza básica)
    .withColumn("motivo", trim(col("motivo").cast("string")))
    
    # 3. CIRURGIA DE RECONSTRUÇÃO (Regex)
    # O ponto (.) substitui o caractere estragado. 
    
    .withColumn("motivo", regexp_replace(col("motivo"), "Contrata..o", "Contratacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Contesta..o", "Contestacao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Altera..o", "Alteracao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "cart.o", "cartao"))
    .withColumn("motivo", regexp_replace(col("motivo"), "D.vidas", "Duvidas"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Informa..es", "Informacoes"))
    .withColumn("motivo", regexp_replace(col("motivo"), "Solicita..o", "Solicitacao"))

    # \\b significa "borda da palavra". Só pega se começar e terminar ali.
    .withColumn("motivo", regexp_replace(col("motivo"), "(?i)\\bn.o\\b", "Nao"))
    
    # 4. Padronização Visual (Capitalize)
    # Deixa "duvidas gerais" -> "Duvidas Gerais"
    .withColumn("motivo", initcap(col("motivo")))
    
    # 5. Tratamento de Nulos
    # Regra: Motivo vazio vira "Motivo Nao Informado"
    .fillna("Motivo Nao Informado", subset=["motivo"])
    .withColumn("motivo", 
                when((col("motivo") == "") | (col("motivo").isNull()), "Motivo Nao Informado")
                .otherwise(col("motivo")))
)

display(df_motivo_tratado)

### 3.5. Padronização de Canais
No dataframe `df_canal_tratado`, normalizei a escrita dos canais de atendimento (unificando "U.r.a" e "URA").
Implementei uma lógica hierárquica de regras (`when/otherwise`), priorizando a identificação de "Atendimento Especializado" antes de "Inicial" para evitar erros de classificação por *substrings*.

In [0]:
df_canal_tratado = (
    df_motivo_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c3", "canal")
    
    # 2. Tipagem e Trim
    .withColumn("canal", trim(col("canal").cast("string")))
    
    # 3. NORMALIZAÇÃO E PADRONIZAÇÃO
    .withColumn("canal", 
                
                # Regra 1: Chatbot
                when(lower(col("canal")).like("%chat%"), "Chatbot")
                
                # Regra 2: URA (Pega URA ou U.r.a)
                .when((lower(col("canal")).like("%ura%")) | (lower(col("canal")).like("%u.r.a%")), "URA")
                
                # Regra 3: Web e Email (Adicionei conforme sua lista)
                .when(lower(col("canal")).like("%web%"), "Web")
                .when(lower(col("canal")).like("%mail%"), "Email")
                
                # Regra 4: ATENDIMENTO ESPECIALIZADO (Checa ANTES do Inicial)
                # Se tiver a palavra "especializado" em qualquer lugar, classifica aqui
                .when(lower(col("canal")).like("%especializ%"), "Atendimento Especializado")
                
                # Regra 5: ATENDIMENTO INICIAL
                # Pega "Inicial", "Atend. Inicial", ou qualquer "Atend" genérico que sobrou
                .when((lower(col("canal")).like("%inici%")) | (lower(col("canal")).like("%atend%")), "Atendimento Inicial")
                
                .otherwise(initcap(col("canal")))
               )

    # 4. Tratamento de Nulos
    .fillna("Canal Nao Identificado", subset=["canal"])
    .withColumn("canal", 
                when((col("canal") == "") | (col("canal").isNull()), "Canal Nao Identificado")
                .otherwise(col("canal")))
)

# Validação Final
print("Validação: Verifique se Especializado e Inicial estão separados:")
df_canal_tratado.groupBy("canal").count().show(truncate=False)

display(df_canal_tratado)

%md
### 3.6. Flag Binária de Resolução
No dataframe `df_resolvido_tratado`, transformei a coluna de status em uma flag binária limpa.
Em vez de tratar acentos individualmente, usei uma lógica robusta que verifica a presença das letras "s" ou "n" (`like %s%`), blindando o código contra variações de escrita como "Sim", "SIM" ou erros de caracteres no "Não".

In [0]:
df_resolvido_tratado = (
    df_canal_tratado # Continua do passo anterior (Canal)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c4", "resolvido")
    
    # 2. Tipagem e Trim
    .withColumn("resolvido", trim(col("resolvido").cast("string")))
    
    # 3. NORMALIZAÇÃO BINÁRIA (melhor prática)
    # Estratégia: Em vez de brigar com o acento, usamos a lógica do "Contém S"
    .withColumn("resolvido", 
                
                # Regra 1: Se tiver "s" ou "S" (Sim, S, yes), vira "Sim"
                when(lower(col("resolvido")).like("%s%"), "Sim")
                
                # Regra 2: Se tiver "n" ou "N" (Nao, No, No), vira "Nao"
                .when(lower(col("resolvido")).like("%n%"), "Nao")
                
                # Caso contrário (Vazio ou Lixo), vira "Nao Informado"
                .otherwise("Nao Informado")
               )

    # 4. Tratamento de Nulos (Garantia Extra)
    # Se sobrar algum null real, vira "Nao Informado"
    .fillna("Nao Informado", subset=["resolvido"])
)

# Validação: Deve aparecer APENAS "Sim", "Nao" e talvez "Nao Informado"
print("Distribuição da coluna Resolvido:")
df_resolvido_tratado.groupBy("resolvido").count().show()

display(df_resolvido_tratado)

### 3.7. Estruturação da Hora de Abertura
Nesta etapa, tratei exclusivamente a coluna `hora_abertura_chamado` (antiga `_c5`).
Embora os dados atuais estivessem vazios ou nulos, decidi forçar a tipagem imediata para `Timestamp` (*Schema Enforcement*). Essa decisão garante que a tabela Silver nasça com a estrutura correta de "Data e Hora" para receber dados futuros, evitando que a coluna permaneça como um texto genérico indefinido.

In [0]:
df_hora_abertura_tratado = (
    df_resolvido_tratado # Continua do passo anterior
    
    # 1. Renomear (Snake Case e Descritivo)
    .withColumnRenamed("_c5", "hora_abertura_chamado")
    
    # 2. Tipagem Forte (Schema Enforcement)
    # Mesmo que esteja tudo Null ou vazio, o tipo é Timestamp.
    .withColumn("hora_abertura_chamado", col("hora_abertura_chamado").cast("timestamp"))
)

# Validação (quero ver o schema como "timestamp" e os dados como "null")
print("Schema da coluna:")
df_hora_abertura_tratado.select("hora_abertura_chamado").printSchema()

print("\nVisualização dos dados (Devem estar null):")
df_hora_abertura_tratado.select("hora_abertura_chamado").show(5)

display(df_hora_abertura_tratado)

### 3.8. Estruturação Temporal e Regra de Cópia
No dataframe `df_inicio_tratado`, forcei a tipagem das colunas de horário para `Timestamp`.
Implementei a regra de negócio para a **Hora de Início**: quando o registro indicava "igual a hora de abertura", o código copiou dinamicamente o valor da coluna anterior.

In [0]:
df_inicio_tratado = (
    df_hora_abertura_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c6", "hora_inicio_atendimento")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("hora_inicio_atendimento", trim(col("hora_inicio_atendimento")))
    
    # 3. Lógica de Negócio (A cópia condicional)
    # Se o texto disser "igual...", ele busca o valor da coluna hora_abertura_chamado.
    .withColumn("hora_inicio_atendimento", 
                when(lower(col("hora_inicio_atendimento")).like("%igual%"), col("hora_abertura_chamado"))
                .otherwise(col("hora_inicio_atendimento")))
    
    # 4. Tipagem Final (Schema Enforcement)
    # Tudo que não for data válida vira Null automaticamente aqui
    .withColumn("hora_inicio_atendimento", col("hora_inicio_atendimento").cast("timestamp"))
)

# Validação:
print("Schema atualizado:")
df_inicio_tratado.select("hora_inicio_atendimento").printSchema()

display(df_inicio_tratado)

### 3.9. Estruturação da Hora de Finalização
Nesta etapa, tratei a coluna `hora_finalizacao_atendimento` (antiga `_c7`).
Realizei a limpeza de espaços em branco (*trim*) e forcei a conversão direta para o tipo `Timestamp`. Diferente da hora de início, não houve necessidade de regras condicionais complexas, mas a definição estrita do tipo garante que a tabela esteja tecnicamente preparada para receber os registros de tempo assim que estiverem disponíveis na origem.

In [0]:
df_fim_tratado = (
    df_inicio_tratado # Continua do passo anterior
    
    # 1. Renomear
    .withColumnRenamed("_c7", "hora_finalizacao_atendimento")
    
    # 2. Trim (Limpeza básica de espaços invisíveis)
    .withColumn("hora_finalizacao_atendimento", trim(col("hora_finalizacao_atendimento")))
    
    # 3. Tipagem (Schema Enforcement)
    # Transforma texto/vazio em Data Real.
    .withColumn("hora_finalizacao_atendimento", col("hora_finalizacao_atendimento").cast("timestamp"))
)

# Validação do Schema
print("Schema final das colunas de tempo:")
df_fim_tratado.select("hora_abertura_chamado", 
                      "hora_inicio_atendimento", 
                      "hora_finalizacao_atendimento").printSchema()

display(df_fim_tratado)

### 3.10. Sanitização do Tempo de Espera
No dataframe `df_espera_tratado`, tratei a métrica `tempo_espera_segundos`.
Identifiquei que a origem enviava a string "NULL", então apliquei uma sanitização para converter esse texto em nulo real antes da tipagem para `int`. Assumi valores nulos como `0` e corrigi eventuais números negativos para garantir a consistência dos cálculos de média futuros.

In [0]:
df_espera_tratado = (
    df_fim_tratado # Continua do passo anterior
    
    # 1. Renomear (coloquei _segundos pra especificar o tipo de tempo)
    .withColumnRenamed("_c8", "tempo_espera_segundos")
    
    # 2. Trim (Limpeza de espaços)
    .withColumn("tempo_espera_segundos", trim(col("tempo_espera_segundos")))
    
    # 3. SANITIZAÇÃO 
    # Antes de converter para número, removi a palavra "NULL" e vazios
    .withColumn("tempo_espera_segundos", 
                when((col("tempo_espera_segundos") == "NULL") | (col("tempo_espera_segundos") == ""), None)
                .otherwise(col("tempo_espera_segundos")))
    
    # 4. Tipagem
    .withColumn("tempo_espera_segundos", col("tempo_espera_segundos").cast("int"))
    
    # 5. Negativos viram 0
    .withColumn("tempo_espera_segundos", 
                when(col("tempo_espera_segundos") < 0, 0)
                .otherwise(col("tempo_espera_segundos")))
    
    # Nulos viram 0 para cálculo de média
    .fillna(0, subset=["tempo_espera_segundos"])
)

# Validação
print("Estatísticas do Tempo de Espera (Segundos):")
df_espera_tratado.select("tempo_espera_segundos").describe().show()

display(df_espera_tratado)

### 3.11. Sanitização do Tempo de Conversa
No dataframe `df_conversa_tratado`, apliquei a mesma lógica de limpeza na coluna `tempo_conversa_segundos`.
Mantive os registros com duração "1" (mesmo sem *timestamps* válidos), preservando a informação de chamadas rápidas para análises de "Chamadas Fantasmas" na camada Gold.

In [0]:
df_conversa_tratado = (
    df_espera_tratado # Continua do passo anterior (Espera)
    
    # 1. Renomear
    .withColumnRenamed("_c9", "tempo_atendimento_segundos")
    
    # 2. Trim
    .withColumn("tempo_atendimento_segundos", trim(col("tempo_atendimento_segundos")))
    
    # 3. SANITIZAÇÃO (O Fix do "NULL" string)
    # Removemos a palavra escrita "NULL" antes de converter
    .withColumn("tempo_atendimento_segundos", 
                when((col("tempo_atendimento_segundos") == "NULL") | (col("tempo_atendimento_segundos") == ""), None)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # 4. Tipagem (Integer)
    .withColumn("tempo_atendimento_segundos", col("tempo_atendimento_segundos").cast("int"))
    
    # 5. Regras de Sanidade
    # Negativos viram 0
    .withColumn("tempo_atendimento_segundos", 
                when(col("tempo_atendimento_segundos") < 0, 0)
                .otherwise(col("tempo_atendimento_segundos")))
    
    # Nulos viram 0 (Assumo zero conversa se estiver vazio)
    .fillna(0, subset=["tempo_atendimento_segundos"])
)

# Validação
print("Estatísticas do Tempo de Conversa (Segundos):")
df_conversa_tratado.select("tempo_atendimento_segundos").describe().show()

display(df_conversa_tratado)

### 3.12. ID Atendente e Carga Final
No dataframe `df_final`, tratei a coluna `id_atendente`. Preenchi os valores nulos com `-1`, criando a categoria "Atendimento Automático" para evitar perdas em cruzamentos com a tabela de funcionários.
Por fim, gravei o resultado na tabela `chamados` do banco `silver_db_name`, utilizando o formato Delta com sobrescrita (`mode("overwrite")`) para atualizar a camada Silver.

In [0]:
df_final = (
    df_conversa_tratado # Continua do passo anterior (Tempo Conversa)
    
    # 1. Renomear (Snake Case)
    .withColumnRenamed("_c10", "id_atendente")
    
    # 2. Trim
    .withColumn("id_atendente", trim(col("id_atendente")))
    
    # 3. SANITIZAÇÃO (Limpa a string "NULL" e vazios)
    .withColumn("id_atendente", 
                when((col("id_atendente") == "NULL") | (col("id_atendente") == ""), None)
                .otherwise(col("id_atendente")))
    
    # 4. Tipagem (Integer)
    .withColumn("id_atendente", col("id_atendente").cast("int"))
    
    # 5. Tratamento de Nulos
    # Se estiver vazio, coloquei -1 (Indica URA/Bot ou erro de sistema)
    .fillna(-1, subset=["id_atendente"])
)

# Validação
print("Amostra dos IDs de Atendente:")
df_final.select("id_atendente").distinct().show(10)

# --- GRAVAÇÃO FINAL DA TABELA SILVER ---
nome_tabela_silver = f"{catalogo}.{silver_db_name}.ft_chamados"

(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_tabela_silver)
)

print(f"Tabela salva em: {nome_tabela_silver}")
display(df_final)

## 4. Tratamento da tabela `custos`

### Descrição
Este notebook realiza a transformação de dados da camada **Bronze** para **Silver** da tabela `custos` (origem) para `vcredit_custos` (destino).

### Adaptação de Schema
Como a ingestão Bronze foi realizada sem cabeçalho, as colunas originais (`_c0`, `_c1`, `_c2`) serão renomeadas para nomes de negócio (`id_custo`, `id_chamado`, `custo`) logo no início do processamento.

### Objetivos
* Renomear colunas técnicas para nomes de negócio
* Padronizar a coluna `custo` (remover texto "reais" e corrigir separadores)
* Converter tipo de dados de String para Decimal
* Garantir unicidade pela chave primária
* Salvar na camada Silver em formato Delta Lake

In [0]:
tabela_origem = "ft_custos" 

print(f"🚀 Carregando tabela Bronze: {catalogo}.{bronze_db_name}.{tabela_origem}")

# 1. Carregar a tabela bruta
df_bronze_raw = spark.read.table(f"{catalogo}.{bronze_db_name}.{tabela_origem}")

# 2. Renomear colunas genéricas para nomes de negócio
# _c0 -> id_custo
# _c1 -> id_chamado
# _c2 -> custo
df_bronze = df_bronze_raw \
    .withColumnRenamed("_c0", "id_custo") \
    .withColumnRenamed("_c1", "id_chamado") \
    .withColumnRenamed("_c2", "custo")

display(df_bronze.limit(5))

### Análise Exploratória
Agora com as colunas renomeadas, analisamos o schema e uma amostra dos dados para confirmar os padrões de sujeira na coluna de valor.

In [0]:
print(f"Total de registros: {df_bronze.count()}")

# Schema e amostra
df_bronze.printSchema()

# Verificando padrões na coluna 'custo' (ex: '0.0026reais' vs '0,1816')
print("Amostra da coluna 'custo' (dados brutos):")
display(df_bronze.select("custo").sample(withReplacement=False, fraction=0.1).limit(10))

### Problemas Identificados
1.  **Formato Inconsistente:** A coluna `custo` mistura formatos. Alguns registros possuem o sufixo "reais" e ponto, outros usam vírgula como decimal.
2.  **Tipo Incorreto:** Dados numéricos estão tipados como `string`.
3.  **Nomenclatura:** O nome `custo` é genérico, vamos alterar para `valor_custo`.

---
### Transformações e Limpeza
Aplicamos a limpeza agressiva (Regex) para remover textos e padronizar o formato numérico, além de ajustar a tipagem.

In [0]:
# Tratamento da coluna de valor
df_step1 = df_bronze.select(
    F.col("id_custo"),
    F.col("id_chamado"),
    
    # Remove tudo que não for número/ponto/vírgula, troca ',' por '.' e converte
    F.regexp_replace(
        F.regexp_replace(F.col("custo"), "[^0-9,.]", ""), 
        ",", "."
    ).cast(DecimalType(18, 6)).alias("valor_custo"),
    
    # Mantendo a coluna de controle da equipe
    F.col("data_ingestao").alias("ingestion_timestamp")
)

print("Amostra após limpeza:")
display(df_step1.limit(10))
df_step1.printSchema()

### Tratamento de Duplicatas e Ordenação
Garantia de integridade da chave primária.

In [0]:
# --- Passo 2: Tratamento de Duplicatas ---
# Contagem antes
total_antes = df_step1.count()

# Remover duplicatas pelo ID (Chave Primária)
# Criamos o df_silver final a partir daqui
df_silver = df_step1.dropDuplicates(["id_custo"])

# Contagem depois
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

# --- Passo 3: Ordenação ---
df_silver = df_silver.orderBy(F.col("id_custo"))

display(df_silver.limit(10))

### Análise Completa de Qualidade - Custos
Agora que os dados estão limpos, calculamos métricas de qualidade e estatísticas financeiras para garantir que não perdemos dados importantes e que os valores fazem sentido.

In [0]:
# Análise completa de qualidade e estatísticas financeiras
qualidade_dados = df_silver.select([
    F.count("*").alias("total_registros"),
    F.countDistinct("id_custo").alias("id_custo_unicos"),
    F.countDistinct("id_chamado").alias("id_chamado_unicos"),
    F.avg("valor_custo").alias("custo_medio"),
    F.sum("valor_custo").alias("custo_total"),
    F.min("valor_custo").alias("custo_minimo"),
    F.max("valor_custo").alias("custo_maximo")
]).collect()[0]

print("RELATÓRIO DE QUALIDADE:")
print(f"Total de registros: {qualidade_dados['total_registros']}")
print(f"IDs custo únicos: {qualidade_dados['id_custo_unicos']}")
print(f"IDs chamado únicos: {qualidade_dados['id_chamado_unicos']}")
print("-" * 30)
print("ESTATÍSTICAS FINANCEIRAS:")
print(f"Custo Médio: R$ {qualidade_dados['custo_medio']:.2f}")
print(f"Custo Mínimo: R$ {qualidade_dados['custo_minimo']:.2f}")
print(f"Custo Máximo: R$ {qualidade_dados['custo_maximo']:.2f}")
print(f"Investimento Total Monitorado: R$ {qualidade_dados['custo_total']:.2f}")

# Verificar integridade 
nulos = df_silver.filter(F.col("id_chamado").isNull()).count()
print("-" * 30)
print(f"Registros órfãos (sem id_chamado): {nulos}")

In [0]:
%sql
DROP TABLE IF EXISTS medalhao_credit.silver_credit.vcredit_custos;

In [0]:
# Salvando na silver
tabela_destino = f"{catalogo}.{silver_db_name}.ft_custos"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_destino)

print(f"✅ Tabela {tabela_destino} salva com sucesso!")

In [0]:
display(df_silver.limit(5))

# Documentação da tabela chamados_geral

## 3. Leitura das Tabelas da Camada Silver

Nesta etapa, carregamos todas as tabelas necessárias para a construção da nossa visão consolidada. Inicialmente, listamos as entidades envolvidas, identificando (nos comentários) as chaves primárias ou estrangeiras que seriam fundamentais para os cruzamentos.

Em seguida, instanciamos cada tabela da camada `silver_credit` em seu próprio DataFrame do Spark. Trouxemos para a memória tanto as tabelas dimensionais (como `base_atendentes`, `clientes`, `canais` e `base_motivos`) quanto as tabelas fato (`chamados`, `custos`, `pesquisa_satisfacao`), preparando o ambiente para a execução dos *joins* e o enriquecimento dos dados.

In [0]:
# leitura das tabelas

nomes_tabelas = [
  "base_atendentes", # id_atendente
  "base_motivos", # nome_motivo
  "canais", # nome_canal
  "chamados", # tabela usada como base para os joins
  "chamados_hora", # id_chamado
  "clientes", # id_cliente
  "custos", # id_chamado
  "pesquisa_satisfacao" # id_chamado
]

df_base_atendentes = spark.table(f"{catalogo}.{silver_db_name}.base_atendentes")
df_base_motivos = spark.table(f"{catalogo}.{silver_db_name}.base_motivos")
df_canais = spark.table(f"{catalogo}.{silver_db_name}.canais")
df_chamados = spark.table(f"{catalogo}.{silver_db_name}.chamados")
df_chamados_hora = spark.table(f"{catalogo}.{silver_db_name}.chamados_hora")
df_clientes = spark.table(f"{catalogo}.{silver_db_name}.clientes")
df_custos = spark.table(f"{catalogo}.{silver_db_name}.custos")
df_pesquisa_satisfacao = spark.table(f"{catalogo}.{silver_db_name}.pesquisa_satisfacao")

## 4. Visualização Amostral da Base de Motivos

Como etapa de validação imediata (*sanity check*), nós executámos uma visualização rápida do DataFrame `df_base_motivos`.

Utilizamos o método `.limit(5)` para restringir a consulta às primeiras cinco linhas, garantindo uma resposta rápida do *cluster*, e o comando `.display()` para renderizar os dados tabularmente. Esta ação permitiu-nos confirmar visualmente se a estrutura e o conteúdo da tabela dimensional de motivos foram carregados corretamente antes de avançarmos para os cruzamentos de dados.

In [0]:
df_base_motivos.limit(5).display()

## 5. Consolidação da Visão Geral de Chamados

Nesta etapa central do pipeline, realizamos a construção do DataFrame `df_chamados_geral`, que serve como a nossa "Tabela Unificada" (*One Big Table*) para análises.

**Estratégia de Joins:**
Utilizamos a tabela `df_chamados` como ponto de partida, aplicando um `right join` com `df_chamados_hora` para garantir que a granularidade temporal fosse preservada como a espinha dorsal do *dataset*. Em seguida, enriquecemos esses dados através de múltiplos `left joins` com as tabelas dimensionais (`atendentes`, `motivos`, `canais`, `clientes`) e tabelas satélites (`custos`, `pesquisa_satisfacao`), consolidando métricas e descrições num único local.

**Seleção e Tratamento de Colunas:**
Durante a projeção dos campos finais (`select`):
* Resolvemos ambiguidades de colunas presentes em múltiplas tabelas (como `id_cliente`), especificando explicitamente a origem através de *aliases*.
* Mapeamos colunas de negócio, observando (conforme comentado no código) que campos como `categoria` e `criticidade` retornaram valores nulos nesta carga, mas foram mantidos para preservar o esquema.
* Adicionamos a coluna `data_criacao_silver` com o *timestamp* atual para rastreabilidade da geração desta tabela consolidada.

Por fim, ordenamos o resultado por `id_chamado` para facilitar a leitura sequencial e exibimos o DataFrame consolidado.

In [0]:
# criacao da tabela chamados_geral

df_chamados_geral = (
    df_chamados
    .join(df_chamados_hora, "id_chamado", "right")
    .join(df_base_atendentes, "id_atendente", "left")
    .join(df_base_motivos, df_chamados["motivo"] == df_base_motivos["nome_motivo"], "left")
    .join(df_canais, df_chamados["canal"] == df_canais["nome_canal"], "left")
    .join(df_clientes, "id_cliente", "left")
    .join(df_custos, "id_chamado", "left")
    .join(df_pesquisa_satisfacao, "id_chamado", "left")
    .select(
        "id_chamado",
        df_chamados["id_cliente"].alias("id_cliente"),
        "motivo",
        "categoria", # tudo nulo
        "categoria_nota",
        "nota_atendimento",
        "criticidade", # tudo nulo
        "canal",
        "status_canal",
        "resolvido",
        df_chamados_hora["hora_abertura_chamado"],
        df_chamados_hora["hora_inicio_atendimento"],
        df_chamados_hora["hora_finalizacao_atendimento"],
        "tempo_espera_segundos",
        "tempo_atendimento_segundos",
        "id_atendente",
        "nome_atendente",
        "nivel_atendimento",
        "valor_custo",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        current_timestamp().alias("data_criacao_silver")
    ).orderBy('id_chamado')
)

display(df_chamados_geral)

## 6. Persistência da Tabela Consolidada (`chamados_geral`)

Como etapa final deste *pipeline*, materializamos o DataFrame `df_chamados_geral` no armazenamento físico do Data Lake.

Utilizamos o comando `.saveAsTable()` direcionando para o catálogo e esquema configurados (`silver_credit`). Optamos pelo modo de escrita `overwrite` (sobrescrita), uma decisão de projeto que assegura que a tabela final reflita exatamente o estado atual do processamento, eliminando resíduos de execuções anteriores e garantindo a consistência da nossa "Single Source of Truth" para as análises futuras.

In [0]:
df_chamados_geral.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_geral')